In [1]:
# 单元 1：基础导入与配置
import os
import sqlite3
import pandas as pd
import numpy as np

def calculate_tet(filepath, time_step=0.1):  
    try:  
        conn = sqlite3.connect(filepath)  
        
        df = pd.read_sql_query("SELECT * FROM trajectory_data", conn)  
        conn.close()  
        
        if 'frame' not in df.columns:  
            print(f"警告：{filepath} 缺少 'frame' 字段")  
            return np.nan  
        
        tet = df['frame'].max() * time_step
        
        return tet  
    
    except Exception as e:  
        print(f"处理 {filepath} 时发生错误: {e}")  
        return np.nan  

def process_small_tet():  
    base_path = "Small_Average"  
    layouts = ["A", "B", "C", "A2", "B2", "C2"]  
    widths = [round(w * 0.2 + 1.0, 1) for w in range(0, 8)]  # 1.0 - 2.4 m  
    runs = list(range(5))  
    
    results = {layout: {} for layout in layouts}  
    
    for layout in layouts:  
        layout_path = os.path.join(base_path, layout)  
        for width in widths:  
            tet_runs = []  
            for run in runs:
                filename = f"Width_{width:.1f}_{run}.sqlite"  
                filepath = os.path.join(layout_path, filename)  
                
                if os.path.exists(filepath):  
                    tet = calculate_tet(filepath)  
                    tet_runs.append(tet)  
                else:  
                    print(f"文件不存在: {filepath}")  
            
            # 计算该 width 下的 TET 均值  
            results[layout][width] = np.mean(tet_runs) if tet_runs else np.nan  
    
    # 转换为 DataFrame  
    df_tet = pd.DataFrame(results).round(2)  
    
    # 重新设置索引为宽度值  
    df_tet.index = widths  
    
    # 保存 CSV  
    os.makedirs("data/processed", exist_ok=True)  
    df_tet.to_csv("data/processed/small_tet_mean.csv")  
    
    return df_tet  

In [2]:

# 单元 4：执行并展示结果  
df_result = process_small_tet()  
print(df_result)  

         A      B      C     A2     B2     C2
1.0  38.22  42.30  33.70  19.72  19.68  16.58
1.2  30.42  37.00  30.74  17.04  19.94  15.22
1.4  27.80  29.96  27.22  16.52  18.70  14.46
1.6  26.42  26.66  28.04  16.02  19.00  14.34
1.8  25.28  23.46  25.66  15.80  19.26  14.42
2.0  22.74  23.24  25.38  15.66  18.66  14.34
2.2  21.50  23.22  24.96  15.48  17.80  13.72
2.4  21.54  22.94  21.38  15.50  17.52  13.84


In [5]:
def process_big_tet():  
    base_path = "Big_Average"  
    layouts = ["Exit"]  
    widths1 = [round(w * 0.2 + 1.0, 1) for w in range(0, 4)]  # 1.0 - 1.6 m  
    widths2 = [round(w * 0.2 + 1.0, 1) for w in range(0, 4)]  # 1.0 - 1.6 m  
    runs = list(range(5))
    
    results = {layout: {} for layout in layouts}  
    
    for layout in layouts:  
        layout_path = os.path.join(base_path, layout)  
        for width1 in widths1:  \
            for width2 in widths2:
                tet_runs = []  
                for run in runs:
                    filename = f"Width_{width:.1f}_{run}.sqlite"  
                    filepath = os.path.join(layout_path, filename)  
                    
                    if os.path.exists(filepath):  
                        tet = calculate_tet(filepath)  
                        tet_runs.append(tet)  
                    else:  
                        print(f"文件不存在: {filepath}")  
                
                # 计算该 width 下的 TET 均值  
                results[layout][width1][width2] = np.mean(tet_runs) if tet_runs else np.nan  
    
    # 转换为 DataFrame  
    df_tet = pd.DataFrame(results).round(2)  
    
    # 重新设置索引为宽度值
    df_tet.index = widths
    
    # 保存 CSV  
    os.makedirs("data/processed", exist_ok=True)  
    df_tet.to_csv("data/processed/Multi_big_tet_mean.csv")  
    
    return df_tet  

SyntaxError: invalid syntax (3284869147.py, line 13)

In [4]:
df = process_big_tet() 
print(df)  

      Exit
1.0  84.86
1.2  74.86
1.4  72.92
1.6  70.22


In [20]:
def process_multi_big_tet():  
    base_path = "Big_Average"  
    layouts = ["MultiRoom"]  
    widths1 = [round(w * 0.2 + 1.0, 1) for w in range(0, 4)]  # 1.0 - 1.6 m  
    widths2 = [round(w * 0.2 + 1.0, 1) for w in range(0, 4)]  # 1.0 - 1.6 m  
    runs = list(range(5))
    
    results = {layout: {} for layout in layouts}  
    
    for layout in layouts:  
        layout_path = os.path.join(base_path, layout)  
        for width1 in widths1:  
            for width2 in widths2:
                tet_runs = []  
                for run in runs:
                    filename = f"Width_{width1:.1f}_{width2:.1f}_{run}.sqlite"  # 文件名格式
                    filepath = os.path.join(layout_path, filename)  
                    
                    if os.path.exists(filepath):  
                        tet = calculate_tet(filepath)  
                        tet_runs.append(tet)  
                    else:  
                        print(f"文件不存在: {filepath}")  
                
                if width1 not in results[layout]:
                    results[layout][width1] = {}
                
                # 计算该宽度下的 TET 均值
                results[layout][width1][width2] = np.mean(tet_runs) if tet_runs else np.nan  
    
    # 转换为 DataFrame
    flattened_results = [
        (layout, width1, width2, results[layout][width1].get(width2, np.nan))
        for layout in results
        for width1 in results[layout]
        for width2 in results[layout][width1]
    ]
    
    # 创建 DataFrame
    df_tet = pd.DataFrame(flattened_results, columns=['Layout', 'Width1', 'Width2', 'Mean_TET'])
    
    # 检查 DataFrame 列数
    print(df_tet.head())  # 输出前几行以检查

    # 保存 CSV  
    os.makedirs("data/processed", exist_ok=True)
    df_tet.to_csv("data/processed/Multi_big_tet_mean.csv", index=False)  
    
    return df_tet

In [21]:
df_multi = process_multi_big_tet() 
print(df_multi)  

      Layout  Width1  Width2  Mean_TET
0  MultiRoom     1.0     1.0     77.00
1  MultiRoom     1.0     1.2     72.32
2  MultiRoom     1.0     1.4     70.98
3  MultiRoom     1.0     1.6     72.60
4  MultiRoom     1.2     1.0     81.32
       Layout  Width1  Width2  Mean_TET
0   MultiRoom     1.0     1.0     77.00
1   MultiRoom     1.0     1.2     72.32
2   MultiRoom     1.0     1.4     70.98
3   MultiRoom     1.0     1.6     72.60
4   MultiRoom     1.2     1.0     81.32
5   MultiRoom     1.2     1.2     74.16
6   MultiRoom     1.2     1.4     69.66
7   MultiRoom     1.2     1.6     70.82
8   MultiRoom     1.4     1.0     87.82
9   MultiRoom     1.4     1.2     74.26
10  MultiRoom     1.4     1.4     71.38
11  MultiRoom     1.4     1.6     70.70
12  MultiRoom     1.6     1.0     82.10
13  MultiRoom     1.6     1.2     80.02
14  MultiRoom     1.6     1.4     71.48
15  MultiRoom     1.6     1.6     70.52
